In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [2]:
# add the csv file to a pandas database
df = pd.read_csv("TrainingPortion.csv")

# QC check to make sure that all categories are there
# print("\ncolumns\n", list(df.columns))
# print("types\n", df.dtypes)

In [3]:
# Once QC is done, remove id (useless for prediction)
if "id" in df.columns:
    df.drop("id", axis = 1, inplace = True)

print (df.shape)

(569, 31)


In [4]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(duplicates)

0


In [5]:
# Number of values benign, and malignant
print(f"\nDiagnosis value counts (before encoding):\n{df['diagnosis'].value_counts()}")

# map each to 1 or 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# Double checking 
print(f"\nDiagnosis value counts (after encoding):\n{df['diagnosis'].value_counts()}")
print(f"Class balance: {df['diagnosis'].mean():.1%} malignant, {1 - df['diagnosis'].mean():.1%} benign")


Diagnosis value counts (before encoding):
diagnosis
B    357
M    212
Name: count, dtype: int64

Diagnosis value counts (after encoding):
diagnosis
0    357
1    212
Name: count, dtype: int64
Class balance: 37.3% malignant, 62.7% benign


In [6]:
# Breaks the fields into the target (want we wanna find) and the feature (what we have)
x = df.drop("diagnosis", axis =1)
y = df["diagnosis"]

print(x.head(3))
print(x.shape)

print(y.head(3))
print(y.shape)

for i, col in enumerate(x.columns):
    print ((i+1), col)

   radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
0        17.99         10.38           122.8     1001.0          0.11840   
1        20.57         17.77           132.9     1326.0          0.08474   
2        19.69         21.25           130.0     1203.0          0.10960   

   compactness_mean  concavity_mean  concave points_mean  symmetry_mean  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   

   fractal_dimension_mean  ...  radius_worst  texture_worst  perimeter_worst  \
0                 0.07871  ...         25.38          17.33            184.6   
1                 0.05667  ...         24.99          23.41            158.8   
2                 0.05999  ...         23.57          25.53            152.5   

   area_worst  smoothness_worst  compactness_worst  concavity_worst  \
0 

In [16]:
# Split the data into the testing portion, and the training portion makes sure malignant and benign ratios are similar in the test and training
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, stratify=y, random_state=42)

print(f"Training set:  {X_train.shape[0]} samples ({X_train.shape[0]/len(x)*100:.1f}%)")
print(f"Test set:      {X_test.shape[0]} samples ({X_test.shape[0]/len(x)*100:.1f}%)")
print(f"\nTraining class balance: {y_train.mean():.1%} malignant")
print(f"Test class balance:     {y_test.mean():.1%} malignant")

Training set:  455 samples (80.0%)
Test set:      114 samples (20.0%)

Training class balance: 37.4% malignant
Test class balance:     36.8% malignant


In [17]:
# scales and generalizes the data so that each aspect affects the algorithm equally
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

In [24]:
# Correlation Check
corr_matrix = X_train_scaled.corr().abs()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [
    (col, upper_tri[col].idxmax(), upper_tri[col].max())
    for col in upper_tri.columns
    if any(upper_tri[col] > 0.9)
]

print("high correlation when r >0.9")
for feat1, feat2, corr in sorted(high_corr_pairs, key= lambda x: x[2], reverse = True):
    print(f"{feat1}, corresponds with {feat2} by {corr:.3f}")

print("for stability and accuracy you may want to drop one of the pairs, or let the model handle it")

high correlation when r >0.9
perimeter_mean, corresponds with radius_mean by 0.998
perimeter_worst, corresponds with radius_worst by 0.994
area_mean, corresponds with radius_mean by 0.987
area_worst, corresponds with radius_worst by 0.983
perimeter_se, corresponds with radius_se by 0.975
radius_worst, corresponds with perimeter_mean by 0.969
area_se, corresponds with radius_se by 0.952
concave points_mean, corresponds with concavity_mean by 0.921
concave points_worst, corresponds with concave points_mean by 0.912
texture_worst, corresponds with texture_mean by 0.912
for stability and accuracy you may want to drop one of the pairs, or let the model handle it
